# Détection visage dans les gravures des romans de jeunesse

## Téléchargements et importations

In [ ]:
# Install the ultralytics package from GitHub
#!pip install git+https://github.com/ultralytics/ultralytics.git@main

  Cloning https://github.com/ultralytics/ultralytics.git (to revision main) to /tmp/pip-req-build-l7_9537z
  Running command git clone --filter=blob:none --quiet https://github.com/ultralytics/ultralytics.git /tmp/pip-req-build-l7_9537z
  Resolved https://github.com/ultralytics/ultralytics.git to commit b10fa7be231820cea668c52af45b25d07ff65968
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.4.23-py3-none-any.whl size=1209326 sha256=8f49a6ee50e8a1effdb937689e406c06c9945234110a056b607e64f67b3bc427
  Stored in directory: /tmp/pip-ephem-wheel-cache-9912cy5o/wheels/bb/0b/bb/3516bf4b0d4c640cfbedf6829f1fa9e8d7dfbcd688eeed795c
Successfully built ultralytics


In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import os

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load the YOLO model
#model = YOLO("yolov8n.pt")  # You can choose other YOLO versions (e.g., yolov8s, yolov8m)

#https://github.com/derronqi/yolov8-face?tab=readme-ov-file
#https://drive.google.com/file/d/1qcr9DbgsX3ryrz2uU8w4Xm3cOrRywXqb/view
folder = "/content/drive/MyDrive/ENC-PSL/Cours_Traitement_Image_Automatique_Option_ENC_2026/Litt-Jeunesse/train12/weights/"
model_name = "best.pt" # model déjà entraîné
path = folder + model_name
model = YOLO(path)

In [ ]:
import os

# Chemin du dossier
folder = "/content/drive/MyDrive/ENC-PSL/Cours_Traitement_Image_Automatique_Option_ENC_2026/Litt-Jeunesse/images_inference"

# Extensions des fichiers images à considérer
image_extensions = (".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".gif")

# Liste uniquement les fichiers image qui ne commencent pas par un point
files = [
    f for f in os.listdir(folder)
    if os.path.isfile(os.path.join(folder, f))
    and f.lower().endswith(image_extensions)
    and not f.startswith(".")
]

# Affiche le nombre de fichiers image valides
print("Number of valid image files:")
print(len(files))



Number of valid image files:
247


## Fonction détection visage + sauvegarde des éléments segmentés

Création d'un dossier par gravure contenants les faces

In [ ]:
def save_faces_proportional(image_path, output_dir="output", margin_face=0):

    # Coupe les visages et les sauvegard
    image = cv2.imread(image_path)
    if image is None:
        print(f"Impossible de lire l'image : {image_path}")
        return

    h, w = image.shape[:2]
    results = model.predict(image, imgsz=[w, h])

    name_no_ext = os.path.splitext(os.path.basename(image_path))[0]
    tableau_name = name_no_ext.split(".")[0]

    # Créer dossier
    tableau_dir = os.path.join(output_dir, tableau_name)
    faces_dir = os.path.join(tableau_dir, "faces")
    os.makedirs(faces_dir, exist_ok=True)

    for i, box in enumerate(results[0].boxes):
        # ---- Bounding box du visage ----
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        confidence = box.conf[0].item()
        confidence_str = f"{int(confidence*100):02d}"

        # Crop visage
        x1_crop = max(0, x1 - margin_face)
        y1_crop = max(0, y1 - margin_face)
        x2_crop = min(w, x2 + margin_face)
        y2_crop = min(h, y2 + margin_face)

        face_crop = image[y1_crop:y2_crop, x1_crop:x2_crop]

        # Sauvegarde
        face_filename = f"{tableau_name}_face{i+1}_cf{confidence_str}.jpg"
        cv2.imwrite(os.path.join(faces_dir, face_filename), face_crop)

    print(f"✅ Traitement terminé pour {tableau_name}")

## Execution de la fonction

In [ ]:
import os

# Dossier des images
input_folder =  "/content/drive/MyDrive/ENC-PSL/Cours_Traitement_Image_Automatique_Option_ENC_2026/Litt-Jeunesse/images_inference"
output_folder = "/content/drive/MyDrive/ENC-PSL/Cours_Traitement_Image_Automatique_Option_ENC_2026/Litt-Jeunesse/output_inference_visages"

image_extensions = (".jpg", ".jpeg", ".png")
files = [f for f in os.listdir(input_folder) if f.lower().endswith(image_extensions)]

for file in files:
    image_path = os.path.join(input_folder, file)
    save_faces_proportional(
        image_path=image_path,
        output_dir=output_folder,
        margin_face=0,
    )



WARNING ⚠️ imgsz=[1647, 1036] must be multiple of max stride 32, updating to [1664, 1056]
0: 672x1056 2 faces, 544.6ms
Speed: 24.9ms preprocess, 544.6ms inference, 38.7ms postprocess per image at shape (1, 3, 672, 1056)
✅ Traitement terminé pour Malot_0

WARNING ⚠️ imgsz=[1664, 1028] must be multiple of max stride 32, updating to [1664, 1056]
0: 672x1056 2 faces, 529.9ms
Speed: 13.4ms preprocess, 529.9ms inference, 1.6ms postprocess per image at shape (1, 3, 672, 1056)
✅ Traitement terminé pour Malot_62

WARNING ⚠️ imgsz=[3011, 1608] must be multiple of max stride 32, updating to [3040, 1632]
0: 896x1632 (no detections), 1435.0ms
Speed: 54.0ms preprocess, 1435.0ms inference, 1.4ms postprocess per image at shape (1, 3, 896, 1632)
✅ Traitement terminé pour elephant_5

WARNING ⚠️ imgsz=[1564, 1816] must be multiple of max stride 32, updating to [1568, 1824]
0: 1568x1376 (no detections), 1182.1ms
Speed: 19.0ms preprocess, 1182.1ms inference, 1.3ms postprocess per image at shape (1, 3, 156